# 02 -- Article-level TCM (the default unit)

Article level is the unit at which Macro F1 is conventionally reported for
this benchmark, and hence the default an analyst inherits (paper §III-C). A
technique is present if it is annotated / predicted **anywhere** in the
article.

Produces the `Article` column of Table IV and the raw / bi-normalised
matrices in `matrices/article/`.

Expected (paper §IV-B): rho_within = +0.328, rho_cross = -0.004.

In [ ]:
import sys; sys.path.insert(0, 'src')
from common import *
from scipy.stats import spearmanr

PARA_IDX = {l: build_para_index(l) for l in LANGS}     # only for the article id list
ARTICLE_IDS = {l: set(PARA_IDX[l].keys()) for l in LANGS}
for l in LANGS:
    assert len(ARTICLE_IDS[l]) == EXPECT_ARTICLES[l]
print('article ids:', {LANG_LABEL[l]: len(ARTICLE_IDS[l]) for l in LANGS})

In [ ]:
# --- nine article-level TCMs, full 23x23, no support floor ---
RAW_ART, BI_ART = {}, {}

print(f"{'config':<18} {'units':>6} {'diag':>7} {'IPF':>5} {'iters':>7}")
print('-' * 48)
for l, m in CONFIGS:
    g = load_gold(l)
    p = load_pred(l, m)
    Yg, Yp, ids = article_multihot(g, p, ARTICLE_IDS[l])
    T, n = compute_tcm(Yg, Yp)
    Tb, it, cv = bi_norm(T)
    RAW_ART[(l, m)], BI_ART[(l, m)] = T, Tb
    np.save(MATRICES / 'article' / f'{l}_{m}_raw.npy', T)
    np.save(MATRICES / 'article' / f'{l}_{m}_bi.npy', Tb)
    print(f'{LANG_LABEL[l]} {METHOD_LABEL[m]:<12} {n:>6} '
          f'{np.trace(T)/T.sum():>6.1%} {"ok" if cv else "FAIL":>5} {it:>7}')
    assert cv, f'{l}/{m}: IPF did not converge at article level'
print('\nSaved to matrices/article/')

In [ ]:
# --- rank correlations over all 253 pairs (23 techniques) ---
V = {k: offdiag_vector(Tb, C) for k, Tb in BI_ART.items()}
assert len(next(iter(V.values()))) == C * (C - 1) // 2 == 253

w = [spearmanr(V[(l, a)], V[(l, b)])[0] for l, a, b in within_pairs()]
c = [spearmanr(V[(a, m)], V[(b, m)])[0] for m, a, b in cross_pairs()]

print('=== within-language, cross-method ===')
for (l, a, b), r in zip(within_pairs(), w):
    print(f'  {LANG_LABEL[l]}  {METHOD_LABEL[a]:<9} vs {METHOD_LABEL[b]:<9}  rho={r:+.3f}')
print('\n=== cross-lingual, same method ===')
for (m, a, b), r in zip(cross_pairs(), c):
    print(f'  {METHOD_LABEL[m]:<9}  {LANG_LABEL[a]}-{LANG_LABEL[b]}  rho={r:+.3f}')

print(f'\nrho_within = {np.mean(w):+.3f} +/- {np.std(w):.3f}   (paper: +0.328)')
print(f'rho_cross  = {np.mean(c):+.3f} +/- {np.std(c):.3f}   (paper: -0.004)')
print(f'positive   = {sum(x > 0 for x in w)}/9 within, {sum(x > 0 for x in c)}/9 cross'
      f'   (paper: 4/9 cross)')

In [ ]:
# --- delta_rho ---
st = delta_rho_stats(w, c)
print(f'delta_rho = {st["delta_rho"]:+.3f}   '
      f'CI [{st["ci_lo"]:+.3f}, {st["ci_hi"]:+.3f}]   perm p = {st["perm_p"]:.3f}')
print('paper §IV-B: +0.332, CI [0.08, 0.56], p = 0.010')

pd.DataFrame({'granularity': ['article'] * 18,
              'group': ['within'] * 9 + ['cross'] * 9,
              'rho': list(w) + list(c)}).to_csv(RESULTS / 'rho_article.csv', index=False)
pd.DataFrame([{k: v for k, v in st.items() if k not in ('boot', 'null')}]
             ).assign(granularity='article', treatment='all'
                      ).to_csv(RESULTS / 'delta_rho_article.csv', index=False)
print('\nSaved rho_article.csv, delta_rho_article.csv')